In [2]:
# Impor pustaka (library) yang dibutuhkan
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings # Tambahan untuk menangani peringatan

# 1. Menyiapkan Data Frame
# Format data: [Alkohol, Rokok, Ganja, Frekuensi/Count]
data = {
    'Alkohol': ['Yes', 'Yes', 'Yes', 'Yes', 'No', 'No', 'No', 'No'],
    'Rokok'  : ['Yes', 'Yes', 'No', 'No', 'Yes', 'Yes', 'No', 'No'],
    'Ganja'  : ['Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No'],
    'Count'  : [911, 538, 44, 456, 3, 43, 2, 279]
}

df = pd.DataFrame(data)

# Mengubah tipe data menjadi kategori agar statsmodels mengenalinya sebagai faktor
df['Alkohol'] = df['Alkohol'].astype('category')
df['Rokok'] = df['Rokok'].astype('category')
df['Ganja'] = df['Ganja'].astype('category')

print("Data Observasi:")
print(df)
print("-" * 50)

# 2. Mendefinisikan Model Loglinear
# Model Loglinear dapat diselesaikan menggunakan Regresi Poisson

# Model 1: Independence Model (A, C, M)
# Hanya efek utama, tidak ada interaksi
formula_indep = "Count ~ Alkohol + Rokok + Ganja"
model_indep = smf.glm(formula=formula_indep, data=df, family=sm.families.Poisson()).fit()

# Model 2: Homogeneous Association (AC, AM, CM)
# Semua efek utama dan semua interaksi 2 arah, TANPA interaksi 3 arah
formula_homogen = "Count ~ Alkohol + Rokok + Ganja + Alkohol:Rokok + Alkohol:Ganja + Rokok:Ganja"
model_homogen = smf.glm(formula=formula_homogen, data=df, family=sm.families.Poisson()).fit()

# Model 3: Saturated Model (ACM)
# Memasukkan semua kemungkinan termasuk interaksi 3 arah
formula_saturated = "Count ~ Alkohol * Rokok * Ganja"

# Menekan peringatan (warnings) karena Model Jenuh pasti memicu "Perfect Separation" 
# dan "divide by zero" akibat nilai derajat bebas residual (df_resid) yang bernilai 0.
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    model_saturated = smf.glm(formula=formula_saturated, data=df, family=sm.families.Poisson()).fit()

# 3. Menampilkan Hasil Goodness of Fit (Deviance/Likelihood Ratio)
def print_model_fit(name, model):
    deviance = model.deviance
    df_resid = model.df_resid
    
    # Penanganan khusus jika df_resid = 0 (Model Jenuh) agar tidak error/warning
    if df_resid == 0:
        p_value = 1.0
    else:
        # P-value dihitung dari distribusi Chi-Square
        from scipy.stats import chi2
        p_value = 1 - chi2.cdf(deviance, df_resid)
    
    print(f"Model: {name}")
    print(f"Deviance (G^2): {deviance:.2f}")
    print(f"Degrees of Freedom (df): {df_resid}")
    print(f"P-Value: {p_value:.4f}")
    
    # Interpretasi (alpha = 0.05)
    if p_value > 0.05:
        print("Kesimpulan: Model FIT dengan data observasi (H0 diterima).\n")
    else:
        print("Kesimpulan: Model TIDAK FIT dengan data observasi (H0 ditolak).\n")

print("\n--- UJI KECOCOKAN MODEL (GOODNESS OF FIT) ---\n")
print_model_fit("(A, C, M) Mutual Independence", model_indep)
print_model_fit("(AC, AM, CM) Homogeneous Association", model_homogen)
print_model_fit("(ACM) Saturated Model", model_saturated)

# 4. Menampilkan Ringkasan Parameter untuk Model Terbaik
print("\n--- RINGKASAN PARAMETER MODEL (AC, AM, CM) ---")
print("Karena model ini paling fit secara parsimoni, kita melihat koefisien interaksinya.")
print(model_homogen.summary())

Data Observasi:
  Alkohol Rokok Ganja  Count
0     Yes   Yes   Yes    911
1     Yes   Yes    No    538
2     Yes    No   Yes     44
3     Yes    No    No    456
4      No   Yes   Yes      3
5      No   Yes    No     43
6      No    No   Yes      2
7      No    No    No    279
--------------------------------------------------

--- UJI KECOCOKAN MODEL (GOODNESS OF FIT) ---

Model: (A, C, M) Mutual Independence
Deviance (G^2): 1286.02
Degrees of Freedom (df): 4
P-Value: 0.0000
Kesimpulan: Model TIDAK FIT dengan data observasi (H0 ditolak).

Model: (AC, AM, CM) Homogeneous Association
Deviance (G^2): 0.37
Degrees of Freedom (df): 1
P-Value: 0.5408
Kesimpulan: Model FIT dengan data observasi (H0 diterima).

Model: (ACM) Saturated Model
Deviance (G^2): -0.00
Degrees of Freedom (df): 0
P-Value: 1.0000
Kesimpulan: Model FIT dengan data observasi (H0 diterima).


--- RINGKASAN PARAMETER MODEL (AC, AM, CM) ---
Karena model ini paling fit secara parsimoni, kita melihat koefisien interaksinya.
  

In [3]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2

# 1. Menyiapkan Data Frame
data = {
    'Gender': ['Female']*8 + ['Male']*8,
    'Location': (['Urban']*4 + ['Rural']*4) * 2,
    'SeatBelt': (['No']*2 + ['Yes']*2) * 4,
    'Injury': (['No', 'Yes'] * 8),
    'Frequency': [7287, 996, 11587, 759, 3246, 973, 6134, 757,
                  10381, 812, 10969, 380, 6123, 1084, 6693, 513]
}

df = pd.DataFrame(data)

# Mengonversi menjadi kategori
for col in ['Gender', 'Location', 'SeatBelt', 'Injury']:
    df[col] = df[col].astype('category')

print("=== DATA KECELAKAAN OBSERVASI ===")
print(df.head())
print("-" * 50)

# Fungsi untuk mengevaluasi dan mencetak hasil model
def evaluate_model(name, model_fit):
    print(f"\nModel: {name}")
    print(f"Deviance (G^2): {model_fit.deviance:.2f}")
    print(f"Degrees of Freedom (df): {model_fit.df_resid}")
    
    p_value = 1 - chi2.cdf(model_fit.deviance, model_fit.df_resid)
    print(f"P-Value: {p_value:.4f}")
    
    if p_value > 0.05:
        print("Kesimpulan: Model FIT (Gagal tolak H0)")
    else:
        print("Kesimpulan: Model KURANG FIT (Tolak H0)")
    
    # Menambahkan kolom prediksi ke data untuk komparasi dengan tabel buku
    df_compare = df.copy()
    df_compare['Fitted'] = model_fit.fittedvalues.round(1)
    
    # Menampilkan 5 baris pertama sebagai perbandingan
    print("\nContoh Perbandingan Aktual vs Fitted (5 baris pertama):")
    print(df_compare[['Gender', 'Location', 'SeatBelt', 'Injury', 'Frequency', 'Fitted']].head())
    print("-" * 50)

# 2. Mendefinisikan Model Loglinear 
# (Regresi Poisson)

# Model A: (GI, GL, GS, IL, IS, LS) - Asosiasi Homogen (Semua 2-way)
formula_A = "Frequency ~ (Gender + Location + SeatBelt + Injury)**2"
model_A = smf.glm(formula=formula_A, data=df, family=sm.families.Poisson()).fit()

# Model B: (GLS, GI, IL, IS) - Model dengan efek GLS dan efek utama ke Injury
formula_B = "Frequency ~ Gender*Location*SeatBelt + Gender:Injury + Location:Injury + SeatBelt:Injury"
model_B = smf.glm(formula=formula_B, data=df, family=sm.families.Poisson()).fit()

# 3. Evaluasi
print("\n=== HASIL UJI KECOCOKAN MODEL (GOODNESS OF FIT) ===")
evaluate_model("(GI, GL, GS, IL, IS, LS)", model_A)
evaluate_model("(GLS, GI, IL, IS)", model_B)

=== DATA KECELAKAAN OBSERVASI ===
   Gender Location SeatBelt Injury  Frequency
0  Female    Urban       No     No       7287
1  Female    Urban       No    Yes        996
2  Female    Urban      Yes     No      11587
3  Female    Urban      Yes    Yes        759
4  Female    Rural       No     No       3246
--------------------------------------------------

=== HASIL UJI KECOCOKAN MODEL (GOODNESS OF FIT) ===

Model: (GI, GL, GS, IL, IS, LS)
Deviance (G^2): 23.35
Degrees of Freedom (df): 5
P-Value: 0.0003
Kesimpulan: Model KURANG FIT (Tolak H0)

Contoh Perbandingan Aktual vs Fitted (5 baris pertama):
   Gender Location SeatBelt Injury  Frequency   Fitted
0  Female    Urban       No     No       7287   7166.4
1  Female    Urban       No    Yes        996    993.0
2  Female    Urban      Yes     No      11587  11748.3
3  Female    Urban      Yes    Yes        759    721.3
4  Female    Rural       No     No       3246   3353.8
--------------------------------------------------

Model: (G